# Stock Prediction Model Training

This notebook documents how the project model is trained from market data rather than downloaded from elsewhere.

It reproduces the same architecture used by the app backend:
- `LSTM(128, return_sequences=True)`
- `LSTM(64)`
- `Dense(25)`
- `Dense(1)`

Running all cells will:
1. Download historical stock data with `yfinance`
2. Build 100-step training sequences from closing prices
3. Train the LSTM model
4. Evaluate it on a held-out validation split
5. Save the trained model as `backend/model/stock_prediction_model.keras`


## Environment

Run this notebook from the `backend/` virtual environment used by the project.

Recommended launch command:

```bash
cd backend
source .venv/bin/activate
jupyter notebook
```


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
import yfinance as yf
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import LSTM, Dense
from tensorflow.keras.models import Sequential

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

pd.set_option('display.max_rows', 10)
pd.set_option('display.max_columns', 20)


In [ ]:
SEQ_LEN = 100
TRAIN_PERIOD = '5y'
TICKERS = [
    'AAPL', 'MSFT', 'GOOGL', 'AMZN', 'META',
    'TSLA', 'NVDA', 'RELIANCE.NS', 'TCS.NS', 'INFY.NS'
]

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
MODEL_OUTPUT = PROJECT_ROOT / 'model' / 'stock_prediction_model.keras'
MODEL_OUTPUT.parent.mkdir(parents=True, exist_ok=True)

print('Project root:', PROJECT_ROOT)
print('Model output:', MODEL_OUTPUT)


## Data Collection

The training data comes from Yahoo Finance through the `yfinance` package. We use only the `Close` column because the deployed backend model also predicts from closing prices only.


In [ ]:
def download_close_prices(ticker: str, period: str = TRAIN_PERIOD) -> pd.Series:
    df = yf.download(ticker, period=period, progress=False)
    if df.empty:
        raise ValueError(f'No data found for {ticker}')

    close_df = df[['Close']].copy()
    if hasattr(close_df.columns, 'levels'):
        close_df.columns = close_df.columns.get_level_values(0)

    series = close_df['Close'].dropna().astype('float32')
    series.name = ticker
    return series


close_series_map = {}
for ticker in TICKERS:
    series = download_close_prices(ticker)
    close_series_map[ticker] = series
    print(f'{ticker}: {len(series)} rows')


In [ ]:
preview_df = pd.concat(close_series_map.values(), axis=1)
preview_df.tail()


## Sequence Preparation

Each training sample uses the previous 100 closing prices to predict the next closing price. This matches the saved model input shape `(None, 100, 1)`.


In [ ]:
def build_sequences_from_series(series: pd.Series, seq_len: int = SEQ_LEN):
    scaler = MinMaxScaler(feature_range=(0, 1))
    values = series.to_numpy(dtype='float32').reshape(-1, 1)
    scaled = scaler.fit_transform(values)

    x_data = []
    y_data = []
    for i in range(seq_len, len(scaled)):
        x_data.append(scaled[i - seq_len:i, 0])
        y_data.append(scaled[i, 0])

    x_data = np.array(x_data, dtype='float32').reshape(-1, seq_len, 1)
    y_data = np.array(y_data, dtype='float32').reshape(-1, 1)
    return x_data, y_data, scaler


sequence_batches = []
target_batches = []
ticker_scalers = {}

for ticker, series in close_series_map.items():
    x_ticker, y_ticker, scaler = build_sequences_from_series(series)
    ticker_scalers[ticker] = scaler
    sequence_batches.append(x_ticker)
    target_batches.append(y_ticker)
    print(f'{ticker}: X={x_ticker.shape}, y={y_ticker.shape}')

X = np.concatenate(sequence_batches, axis=0)
y = np.concatenate(target_batches, axis=0)

print('Combined X shape:', X.shape)
print('Combined y shape:', y.shape)


In [ ]:
split_index = int(len(X) * 0.8)
X_train, X_val = X[:split_index], X[split_index:]
y_train, y_val = y[:split_index], y[split_index:]

print('Train:', X_train.shape, y_train.shape)
print('Validation:', X_val.shape, y_val.shape)


## Model Definition

This architecture is intentionally the same as the model currently used in the app backend.


In [ ]:
model = Sequential([
    LSTM(128, return_sequences=True, input_shape=(SEQ_LEN, 1)),
    LSTM(64),
    Dense(25),
    Dense(1),
])

model.compile(optimizer='adam', loss='mean_squared_error')
model.summary()


## Training

`EarlyStopping` is used so the notebook can be rerun without always training for the full number of epochs.


In [ ]:
callbacks = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
]

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=25,
    batch_size=32,
    callbacks=callbacks,
    verbose=1,
)


In [ ]:
history_df = pd.DataFrame(history.history)
history_df.plot(figsize=(10, 5), title='Training History')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.grid(True)
plt.show()


## Validation Metrics

The model outputs scaled prices, so we compute metrics on the scaled validation split for a quick reproducibility check. You can also inverse-transform per ticker for deeper analysis.


In [ ]:
val_pred = model.predict(X_val, verbose=0)
val_mse = mean_squared_error(y_val, val_pred)
val_rmse = float(np.sqrt(val_mse))
val_r2 = r2_score(y_val, val_pred)

print('Validation MSE :', round(float(val_mse), 6))
print('Validation RMSE:', round(val_rmse, 6))
print('Validation R2  :', round(float(val_r2), 6))


## Example Next-Day Prediction

This cell demonstrates how the trained model predicts the next close for one ticker using the most recent 100-day window.


In [ ]:
example_ticker = TICKERS[0]
example_series = close_series_map[example_ticker]
example_scaler = ticker_scalers[example_ticker]
example_values = example_series.to_numpy(dtype='float32').reshape(-1, 1)
example_scaled = example_scaler.transform(example_values)

last_window = example_scaled[-SEQ_LEN:].reshape(1, SEQ_LEN, 1)
next_scaled = model.predict(last_window, verbose=0)
next_price = example_scaler.inverse_transform(next_scaled)[0][0]

print(f'Example ticker: {example_ticker}')
print('Last close      :', round(float(example_series.iloc[-1]), 2))
print('Predicted next  :', round(float(next_price), 2))


## Save Model

This writes the trained model to the same location used by the Django backend.


In [ ]:
model.save(MODEL_OUTPUT)
print('Saved model to:', MODEL_OUTPUT)
